# MCP completo para estudiantes

Este notebook presenta los conceptos esenciales de MCP con ejemplos practicos en Python.


## Mapa rapido de la clase

Esta seccion resume el recorrido de aprendizaje desde fundamentos hasta ejercicios guiados.


In [1]:
from dataclasses import dataclass
from typing import Any, Dict, Callable
from pathlib import Path
import json

print('Entorno listo para el laboratorio MCP')


Entorno listo para el laboratorio MCP


## 1) Arquitectura MCP en lenguaje simple

MCP conecta un agente con herramientas externas mediante un protocolo estandar y predecible.


In [2]:
@dataclass
class ToolDescriptor:
    name: str
    description: str
    input_schema: Dict[str, Any]


search_docs_descriptor = ToolDescriptor(
    name='search_docs',
    description='Busca contenido en una base documental local',
    input_schema={
        'type': 'object',
        'required': ['query'],
        'properties': {
            'query': {'type': 'string', 'minLength': 2},
            'limit': {'type': 'integer', 'minimum': 1, 'maximum': 5},
        },
    },
)

get_weather_descriptor = ToolDescriptor(
    name='get_weather',
    description='Devuelve temperatura de una ciudad conocida',
    input_schema={
        'type': 'object',
        'required': ['city'],
        'properties': {
            'city': {'type': 'string', 'minLength': 2},
            'unit': {'type': 'string', 'enum': ['C', 'F']},
        },
    },
)

for d in [search_docs_descriptor, get_weather_descriptor]:
    print(f'- {d.name}: {d.description}')


- search_docs: Busca contenido en una base documental local
- get_weather: Devuelve temperatura de una ciudad conocida


In [3]:
def _check_type(value: Any, expected: str) -> bool:
    mapping = {
        'string': str,
        'integer': int,
        'number': (int, float),
        'boolean': bool,
        'object': dict,
        'array': list,
    }
    py_type = mapping.get(expected)
    return isinstance(value, py_type) if py_type else True


def validate_args(args: Dict[str, Any], schema: Dict[str, Any]):
    if schema.get('type') != 'object' or not isinstance(args, dict):
        return False, 'Args deben ser un objeto tipo dict'

    required = schema.get('required', [])
    props = schema.get('properties', {})

    for key in required:
        if key not in args:
            return False, f'Falta argumento obligatorio: {key}'

    for key, value in args.items():
        if key not in props:
            return False, f'Argumento no permitido: {key}'

        rule = props[key]
        expected_type = rule.get('type')
        if expected_type and not _check_type(value, expected_type):
            return False, f"{key} debe ser de tipo {expected_type}"

        if isinstance(value, str) and 'minLength' in rule:
            if len(value) < rule['minLength']:
                return False, f"{key} debe tener al menos {rule['minLength']} caracteres"

        if isinstance(value, int):
            if 'minimum' in rule and value < rule['minimum']:
                return False, f"{key} debe ser >= {rule['minimum']}"
            if 'maximum' in rule and value > rule['maximum']:
                return False, f"{key} debe ser <= {rule['maximum']}"

        if 'enum' in rule and value not in rule['enum']:
            return False, f"{key} debe ser uno de {rule['enum']}"

    return True, 'ok'


print(validate_args({'query': 'mcp', 'limit': 3}, search_docs_descriptor.input_schema))
print(validate_args({'query': 'm'}, search_docs_descriptor.input_schema))


(True, 'ok')
(False, 'query debe tener al menos 2 caracteres')


In [4]:
def search_docs(query: str, limit: int = 3):
    corpus = [
        'MCP define una interfaz estandar para tools.',
        'Los schemas ayudan a validar argumentos antes de ejecutar.',
        'Un servidor MCP puede exponer tools, resources y prompts.',
        'El host usa resultados de tools para responder mejor al usuario.',
        'Siempre maneja errores de red, permisos y validacion.',
    ]
    q = query.lower()
    hits = [line for line in corpus if q in line.lower()]
    return {'ok': True, 'results': hits[:limit], 'count': len(hits)}


def get_weather(city: str, unit: str = 'C'):
    db = {'bogota': 18, 'medellin': 24, 'mexico': 21}
    temp_c = db.get(city.lower())
    if temp_c is None:
        return {'ok': False, 'error': f'Ciudad no soportada: {city}'}
    temp = (temp_c * 9 / 5) + 32 if unit == 'F' else temp_c
    return {'ok': True, 'city': city.title(), 'temp': round(temp, 1), 'unit': unit}


TOOL_REGISTRY: Dict[str, Callable[..., Dict[str, Any]]] = {
    'search_docs': search_docs,
    'get_weather': get_weather,
}

DESCRIPTOR_REGISTRY = {
    'search_docs': search_docs_descriptor,
    'get_weather': get_weather_descriptor,
}


In [5]:
def invoke_tool(tool_name: str, args: Dict[str, Any]) -> Dict[str, Any]:
    # Simula el rol del host: resolver descriptor -> validar -> ejecutar
    descriptor = DESCRIPTOR_REGISTRY.get(tool_name)
    if not descriptor:
        return {'ok': False, 'stage': 'discover', 'error': f'Tool desconocida: {tool_name}'}

    valid, msg = validate_args(args, descriptor.input_schema)
    if not valid:
        return {'ok': False, 'stage': 'validate', 'error': msg}

    fn = TOOL_REGISTRY.get(tool_name)
    if not fn:
        return {'ok': False, 'stage': 'dispatch', 'error': 'Tool no implementada'}

    try:
        payload = fn(**args)
        return {'ok': True, 'stage': 'done', 'data': payload}
    except Exception as exc:
        return {'ok': False, 'stage': 'execute', 'error': str(exc)}


tests = [
    ('search_docs', {'query': 'mcp', 'limit': 2}),
    ('search_docs', {'query': 'x', 'limit': 2}),
    ('search_docs', {'query': 'mcp', 'limit': 10}),
    ('get_weather', {'city': 'Bogota'}),
    ('get_weather', {'city': 'Lima'}),
    ('unknown_tool', {'x': 1}),
]

for name, args in tests:
    print(f'\nCALL {name} args={args}')
    print(json.dumps(invoke_tool(name, args), indent=2, ensure_ascii=False))



CALL search_docs args={'query': 'mcp', 'limit': 2}
{
  "ok": true,
  "stage": "done",
  "data": {
    "ok": true,
    "results": [
      "MCP define una interfaz estandar para tools.",
      "Un servidor MCP puede exponer tools, resources y prompts."
    ],
    "count": 2
  }
}

CALL search_docs args={'query': 'x', 'limit': 2}
{
  "ok": false,
  "stage": "validate",
  "error": "query debe tener al menos 2 caracteres"
}

CALL search_docs args={'query': 'mcp', 'limit': 10}
{
  "ok": false,
  "stage": "validate",
  "error": "limit debe ser <= 5"
}

CALL get_weather args={'city': 'Bogota'}
{
  "ok": true,
  "stage": "done",
  "data": {
    "ok": true,
    "city": "Bogota",
    "temp": 18,
    "unit": "C"
  }
}

CALL get_weather args={'city': 'Lima'}
{
  "ok": true,
  "stage": "done",
  "data": {
    "ok": false,
    "error": "Ciudad no soportada: Lima"
  }
}

CALL unknown_tool args={'x': 1}
{
  "ok": false,
  "stage": "discover",
  "error": "Tool desconocida: unknown_tool"
}


## 2) Patrones clave que debes ensenar

Estos patrones ayudan a invocar tools de forma segura, valida y comprensible para el usuario final.


In [6]:
def answer_user(question: str) -> str:
    q = question.lower()

    if 'clima' in q or 'temperatura' in q:
        city = None
        for candidate in ['bogota', 'medellin', 'mexico']:
            if candidate in q:
                city = candidate
                break
        if not city:
            return 'Puedo consultar clima, pero necesito una ciudad (Bogota, Medellin o Mexico).'

        result = invoke_tool('get_weather', {'city': city, 'unit': 'C'})
        if not result['ok']:
            return f"No pude consultar clima ({result['stage']}): {result['error']}"

        data = result['data']
        if not data.get('ok'):
            return f"No hay dato de clima: {data.get('error', 'error desconocido')}"

        return f"En {data['city']} hay {data['temp']} grados {data['unit']}."

    if 'mcp' in q or 'tool' in q:
        result = invoke_tool('search_docs', {'query': 'MCP', 'limit': 2})
        if result['ok']:
            docs = result['data']['results']
            return 'Resumen rapido:\n- ' + '\n- '.join(docs)
        return f"No pude buscar docs: {result['error']}"

    return 'Pregunta valida. Prueba con clima o con conceptos MCP.'


print(answer_user('Cual es el clima en Bogota?'))
print()
print(answer_user('Dame dos ideas sobre MCP'))


En Bogota hay 18 grados C.

Resumen rapido:
- MCP define una interfaz estandar para tools.
- Un servidor MCP puede exponer tools, resources y prompts.


## 3) Exploracion real (opcional) de descriptores MCP en Cursor

Aqui inspeccionas descriptores reales para relacionar la teoria de MCP con un entorno real de trabajo.


In [7]:
mcps_root = Path.home() / '.cursor' / 'projects'

descriptor_files = []
if mcps_root.exists():
    # Busca tool descriptors en proyectos locales de Cursor
    descriptor_files = list(mcps_root.glob('**/mcps/*/tools/*.json'))

print(f'Descriptores encontrados: {len(descriptor_files)}')

for path in descriptor_files[:5]:
    print('-', path)

if descriptor_files:
    sample = descriptor_files[0]
    data = json.loads(sample.read_text())
    print('\nEjemplo de descriptor:')
    print('name:', data.get('name'))
    print('description:', data.get('description'))
    print('keys:', list(data.keys()))


Descriptores encontrados: 333
- /Users/guane/.cursor/projects/var-folders-t3-97hgmq6x6mg3dybs2fbsfcqr0000gn-T-1e43adfa-686c-4ce5-8832-c24a776fbc97/mcps/cursor-ide-browser/tools/browser_cdp.json
- /Users/guane/.cursor/projects/var-folders-t3-97hgmq6x6mg3dybs2fbsfcqr0000gn-T-1e43adfa-686c-4ce5-8832-c24a776fbc97/mcps/cursor-ide-browser/tools/browser_take_screenshot.json
- /Users/guane/.cursor/projects/var-folders-t3-97hgmq6x6mg3dybs2fbsfcqr0000gn-T-1e43adfa-686c-4ce5-8832-c24a776fbc97/mcps/cursor-ide-browser/tools/browser_lock.json
- /Users/guane/.cursor/projects/var-folders-t3-97hgmq6x6mg3dybs2fbsfcqr0000gn-T-1e43adfa-686c-4ce5-8832-c24a776fbc97/mcps/cursor-ide-browser/tools/browser_click.json
- /Users/guane/.cursor/projects/var-folders-t3-97hgmq6x6mg3dybs2fbsfcqr0000gn-T-1e43adfa-686c-4ce5-8832-c24a776fbc97/mcps/cursor-ide-browser/tools/browser_scroll.json

Ejemplo de descriptor:
name: browser_cdp
description: Send a Chrome DevTools Protocol command to the target browser tab. Do not use

## 4) Ejercicios para estudiantes

Esta seccion propone retos cortos para practicar validacion, invocacion de tools y manejo de errores en MCP.
